# Test 1

In [2]:
import sys
import os
# Add parent directory to sys.path
parent_dir = os.path.abspath("..")
sys.path.append(parent_dir)
from backend.pipeline import (
    process_vocab_word
)

import json
import requests
from backend.constants import (
    ANKI_CONNECT_URL    
)

In [2]:
ANKI_CONNECT_URL

'http://localhost:8765'

In [3]:
def request(action, **params):
    return {'action': action, 'params': params, 'version': 6}

def invoke(action, **params):
    payload = request(action, **params)
    response = requests.post(ANKI_CONNECT_URL, json=payload)
    
    if not response.ok:
        raise Exception(f"Request failed with status code {response.status_code}: {response.text}")
    
    response_json = response.json()
    
    if 'error' not in response_json or 'result' not in response_json:
        raise Exception('Invalid response structure')
    if response_json['error'] is not None:
        raise Exception(response_json['error'])
    
    return response_json['result']

In [22]:
import base64

def store_audio_file(filename):
    abs_path = os.path.abspath(filename)  # Get absolute path of the file
    with open(abs_path, "rb") as f:
        audio_data = base64.b64encode(f.read()).decode("utf-8")  # Encode to Base64 and decode to a string
    
    print(abs_path)
    
    response = invoke("storeMediaFile", filename=filename, path=abs_path)
    print(f"Stored {filename}: {response}")  # Print Anki Connect response

In [4]:
invoke('modelNames')

['01 BASIC',
 '5000 French Words 2.0 (E to F)',
 '5000 French Words 2.0 (E to F) C',
 '5000 French Words 2.0 (F to E)',
 '5000 French Words 2.0 (F to E) C',
 'Arabic Abjad',
 'Basic',
 'Basic (and reversed card)',
 'Basic (optional reversed card)',
 'Basic (type in the answer)',
 'Basic+',
 'Basic-09604',
 'Cloze',
 'Cloze+',
 'Cloze++',
 'Core Japanese Vocabulary Extended',
 'French aspirated h',
 'French Ear Training',
 'French IPA deck',
 'French irregular pronunciation',
 'French phonology',
 'French sentences Read Training',
 'French sentences Speak Training',
 'French Verbs',
 'French vowels comparison',
 'Image Occlusion',
 'Intro card',
 'Japanese-75658',
 'Kanji Radical English',
 'Memrise - 8000+ Most Common Swedish Words - Part 1 (of four) - Swedish',
 'Memrise - 8000+ Most Common Swedish Words - Part 2 (of four) - Swedish',
 'Memrise - 8000+ Most Common Swedish Words - Part 3 (of four) - Swedish',
 'Memrise - 8000+ Most Common Swedish Words - Part 4 (of four) - Swedish',
 '

In [7]:
invoke('deckNames')

['* Navajo',
 '5000 Most Common French Words',
 '5000 Most Common French Words::[1] Main Course',
 '5000 Most Common French Words::[1] Main Course::[a] Option 1: Parisian French Audio',
 '5000 Most Common French Words::[1] Main Course::[a] Option 1: Parisian French Audio::I) French to English (Start here)',
 '5000 Most Common French Words::[1] Main Course::[a] Option 1: Parisian French Audio::II) English to French',
 '5000 Most Common French Words::[1] Main Course::[b] Option 2: Canadian French Audio',
 '5000 Most Common French Words::[1] Main Course::[b] Option 2: Canadian French Audio::I) French to English',
 '5000 Most Common French Words::[1] Main Course::[b] Option 2: Canadian French Audio::II) English to French',
 '5000 Most Common French Words::[A. 1] Irregular Verbs Training',
 '5000 Most Common French Words::[A. 1] Irregular Verbs Training::[a] Option 1: Most Frequent Conjs. Come First',
 '5000 Most Common French Words::[A. 1] Irregular Verbs Training::[a] Option 2: One Verb a

In [5]:
invoke('')

Exception: '' is too short

Failed validating 'minLength' in schema['properties']['action']:
    {'minLength': 1, 'type': 'string'}

On instance['action']:
    ''

In [10]:
invoke('createDeck', deck='test1')

1741653962153

In [11]:
# invoke('deleteDecks', decks="test2", cardsToo=True)

In [12]:
# invoke('modelNames')

In [13]:
# print my vocab mining card type
invoke("modelTemplates", modelName="Vocab Mining")

{'Card 1': {'Front': '<span style="font-size: 50px;">{{Target Word}}</span><br>\n{{Word Audio}}',
  'Back': '{{FrontSide}}\n<hr id=answer>\n<span style="font-size: 35px;">{{Source Translation}}</span><br>\n{{Word Audio}}{{Sentence Audio}}<br>\n<span style="font-size: 25px;">{{Target Sentence}}<br></span>\n{{Source Sentence Translation}}<br>\n'},
 'Card 2': {'Front': '<span style="font-size: 50px;">{{Source Translation}}</span>',
  'Back': '{{FrontSide}}\n\n<hr id=answer>\n<span style="font-size: 35px;">{{Target Word}}</span><br>\n{{Word Audio}}{{Sentence Audio}}<br>\n<span style="font-size: 25px;">{{Target Sentence}}<br></span>\n{{Source Sentence Translation}}<br>\n'}}

In [14]:
invoke("modelFieldNames", modelName="Vocab Mining")

['Target Word',
 'Source Translation',
 'Target Sentence',
 'Source Sentence Translation',
 'Word Audio',
 'Sentence Audio']

In [16]:
# generate test fields
"""
{
    "vocab_word": vocab_word,
    "vocab_translation": vocab_translation,
    "example_sentence": example_sentence,
    "example_sentence_translation": example_sentence_translation,
    "vocab_audio": vocab_audio,
    "example_sentence_translation_audio": example_sentence_audio
}
"""
# todo: infer language based on input?
result = process_vocab_word("制す", target_language="Japanese")


Processing vocabulary word: 制す

Checking if audio files exist...
制す_word.mp3: True
制す_sentence.mp3: True


In [17]:
result

{'vocab_word': '制す',
 'vocab_translation': 'Control',
 'example_sentence': '彼は自己の感情を制すことができました。',
 'example_sentence_translation': 'He was able to control his emotions.',
 'vocab_audio_filename': '制す_word.mp3',
 'example_sentence_translation_audio_filename': '制す_sentence.mp3'}

In [18]:
vocab_word = result["vocab_word"]
vocab_translation = result["vocab_translation"]
example_sentence = result["example_sentence"]
example_sentence_translation = result["example_sentence_translation"]
vocab_audio_filename = result["vocab_audio_filename"]
example_sentence_translation_audio_filename = result["example_sentence_translation_audio_filename"]

In [19]:
store_audio_file(vocab_audio_filename)
store_audio_file(example_sentence_translation_audio_filename)

/Users/sethdonaldson/sourcecode/anki-vocab-generator/notebook/制す_word.mp3
Stored 制す_word.mp3: 制す_word.mp3
/Users/sethdonaldson/sourcecode/anki-vocab-generator/notebook/制す_sentence.mp3
Stored 制す_sentence.mp3: 制す_sentence.mp3


In [20]:
vocab_audio_filename

'制す_word.mp3'

In [21]:
import os

print("Checking if audio files exist...")
print(f"{vocab_audio_filename}: {os.path.exists(vocab_audio_filename)}")
print(f"{example_sentence_translation_audio_filename}: {os.path.exists(example_sentence_translation_audio_filename)}")

Checking if audio files exist...
制す_word.mp3: True
制す_sentence.mp3: True


In [22]:

deck_name = f"test1"
model_name = "Vocab Mining"
fields = {
    'Target Word': vocab_word,
    'Source Translation': vocab_translation,
    'Target Sentence': example_sentence,
    'Source Sentence Translation':example_sentence_translation,
}
audio = [
    {
        "filename": vocab_audio_filename,
        "path": os.path.abspath(vocab_audio_filename),
        "fields": [
            "Word Audio"
        ]
    },
    {
        "filename": example_sentence_translation_audio_filename,
        "path": os.path.abspath(example_sentence_translation_audio_filename),
        "fields": [
            "Sentence Audio"
        ]
    }
]
note = {
    "deckName": deck_name,
    "modelName": model_name,
    "fields": fields,
    "audio": audio,
    "tags": ["stenchtoast"],
    "options": {
            "allowDuplicate": False,
            "duplicateScope": "deck",
            "duplicateScopeOptions": {
                "deckName": deck_name,
                "checkChildren": False,
                "checkAllModels": False
            }
    }
}

In [23]:
note

{'deckName': 'test1',
 'modelName': 'Vocab Mining',
 'fields': {'Target Word': '制す',
  'Source Translation': 'Control',
  'Target Sentence': '彼は自己の感情を制すことができました。',
  'Source Sentence Translation': 'He was able to control his emotions.'},
 'audio': [{'filename': '制す_word.mp3',
   'path': '/Users/sethdonaldson/sourcecode/anki-vocab-generator/notebook/制す_word.mp3',
   'fields': ['Word Audio']},
  {'filename': '制す_sentence.mp3',
   'path': '/Users/sethdonaldson/sourcecode/anki-vocab-generator/notebook/制す_sentence.mp3',
   'fields': ['Sentence Audio']}],
 'tags': ['stenchtoast'],
 'options': {'allowDuplicate': False,
  'duplicateScope': 'deck',
  'duplicateScopeOptions': {'deckName': 'test1',
   'checkChildren': False,
   'checkAllModels': False}}}

In [24]:
# add card to deck
invoke("addNote", note=note)

1742759997575

# Cloze Testing

In [12]:
import sys
import os
# Add parent directory to sys.path
parent_dir = os.path.abspath("..")
sys.path.append(parent_dir)
from backend.pipeline import (
    process_vocab_word
)

import json
import requests
from backend.constants import (
    ANKI_CONNECT_URL 
)

In [13]:
from backend.pipeline import *

In [14]:
import re
from backend.pipeline import *
from backend.pipeline import generate_translation

In [15]:
def extract_cloze(text: str) -> tuple[str, str | None]:
    """
    Given a string containing a cloze like "{{c1::WORD}}…",
    returns a tuple (full_sentence, clozed_word).
    If no cloze is found, returns (original_text, None).
    """
    pattern = re.compile(r"\{\{c\d+::(.*?)\}\}")
    match = pattern.search(text)
    if not match:
        return text, None

    clozed_word = match.group(1)
    full_sentence = pattern.sub(r"\1", text)
    return clozed_word, full_sentence

In [16]:
cloze_text_raw = "{{c1::昨日}}, 友だちと映画を見に行きました。"
cloze_phrase, cloze_text = extract_cloze(cloze_text_raw)

## Translation and Definition

In [17]:

cloze_text

'昨日, 友だちと映画を見に行きました。'

In [19]:
# Now do the definition generation
phrase_translation, text_translation = generate_translation(cloze_phrase, cloze_text, "Japanese", "English")

## Audio Generation

In [20]:
# Generate TTS audio
target_language = "Japanese"
language_code = get_language_code(target_language)
cloze_phrase_audio = generate_audio(cloze_phrase, language_code)
cloze_text_audio = generate_audio(cloze_text, language_code)

In [23]:
store_audio_file(cloze_phrase_audio)
store_audio_file(cloze_text_audio)

OSError: [Errno 63] File name too long: '//OExAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA/OExAAmugIgAMJGmVLBUJrOknRyiMRiMP/G43G43GxWCYbDAoFAoFAoFAoJCcVitGjRitGjQIECBAgQQhOc5znOc0CBAAAAAiILA3d/d0L/f/3d0RERERHc93d3EIgAgQQiCxbu7u7iEAAAAAw8PDw8AAAAAAw8PDw8AAAAAAw8PDw8AwAAADDx/+YeHh48Adhx3/v/wdxkPDw8eAAAAAIw8PfONZKG+BUOAEUBhyJymy1ldOquV2Wcu67r/OExBgk8fogAVowAcLs0sui0ujUqppdNZKGqRQCZotitop9ky5QQxRsmg0HFNpj15StuXGX4atTXUnlnmlkZUlVejjKp5mUSKKjaqqqqZp74j35Zr4Ixp2EYM2id8aQ5780LgEkNtuECu0yLCkzW1Mf/m/tsmKwcbanzNearN700DHHuLk0p3I02FRaN7gEAZ7hunC1ykfibfekldLGYPbm6Jak4EA5ZDEf9jYUFdZc6eJRsnVsr2mrIGQMn8cE/OExDcxTCokAZtgAS05yFRqzm97ztHFUNLLmaftux940qrg9wqWLFCWgw5PfaY81qJydcpAYjug2ZdRnjPudnfDktZZzF+NuOVbtXWsOk6c9TMQstUm9s3KUme+u/tlnzaCndTb5PagvgbN1529tnV6TaLFu07ZmZxubz09Slvmb9lrRX+aW7HXN/KfM+3XviosON

In [ ]:
# get audio figured out
# make an actual card using this flow
# write your backend

In [10]:
note = {
  "deckName": "test1",
  "modelName": "Cloze Vocab",
  "fields": {
    "Cloze Text": cloze_text_raw,
    "Cloze Word": cloze_word,
    "Text Translation": text_translation,
    "Word Translation": phrase_translation,
    
  },
  "audio": [
    {
      "filename": "eiga.mp3",
      "path": "/abs/path/to/eiga.mp3",
      "fields": [
        "Cloze Word Audio"
      ]
    },
    {
      "filename": "sentence.mp3",
      "path": "/abs/path/to/sentence.mp3",
      "fields": [
        "Cloze Text Audio"
      ]
    }
  ],
  "tags": ["free-palestine"],
  "options": {
    "allowDuplicate": False,
    "duplicateScope": "deck",
    "duplicateScopeOptions": {
      "deckName": "test1",
      "checkChildren": False,
      "checkAllModels": False
    }
  }
}

In [12]:
note

{'deckName': 'test1',
 'modelName': 'Cloze Vocab',
 'fields': {'Cloze Sentence': '{{c1::昨日}}, 友だちと{{c2::映画}}を{{c3::見に{{c4::行きました}}}}。',
  'Cloze Word': '映画',
  'Word Translation': 'movie',
  'Sentence Translation': 'Yesterday, I went to see a movie with a friend.'},
 'audio': [{'filename': 'eiga.mp3',
   'path': '/abs/path/to/eiga.mp3',
   'fields': ['Word Audio']},
  {'filename': 'sentence.mp3',
   'path': '/abs/path/to/sentence.mp3',
   'fields': ['Sentence Audio']}],
 'tags': ['auto-generated', 'jp-en', 'mobile-input'],
 'options': {'allowDuplicate': False,
  'duplicateScope': 'deck',
  'duplicateScopeOptions': {'deckName': 'Cloze Mining',
   'checkChildren': False,
   'checkAllModels': False}}}

In [11]:
# add card to deck
invoke("addNote", note=note)

Exception: cannot create note because it is empty